# Практическая задача 3. Тематическое моделирование (sklearn и gensim)

Источник: `materials/notes/disciplines__nlp__laboratory-works__gensim-sklearn-topic-modeling.md`.

На вики 2026 третья по списку практика — «дистилляция и квантование» — **без текста условия**. Эта работа — следующая со ссылкой (тематическое моделирование). Данные — из условия ЛР 5 (архив BigARTM): mlsum/ru parquet, как в `materials/notes/disciplines__nlp__laboratory-works__topic-modeling.md`.

## 1. Загрузка обучающей и тестовой выборок (данные ЛР 5)

Файлы parquet с Hugging Face; кэш вне OneDrive. Берём подвыборку статей, чтобы прогон на CPU укладывался в разумное время.

In [1]:
from __future__ import annotations

import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from gensim import corpora, models
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

CACHE = Path.home() / '.cache' / 'studies2026-nlp' / 'mlsum-ru'
CACHE.mkdir(parents=True, exist_ok=True)

TRAIN_URL = (
    'https://huggingface.co/datasets/mlsum/resolve/refs%2Fconvert%2Fparquet/'
    'ru/train/0000.parquet?download=true'
)
TEST_URL = (
    'https://huggingface.co/datasets/mlsum/resolve/refs%2Fconvert%2Fparquet/'
    'ru/test/0000.parquet?download=true'
)
TRAIN_PATH = CACHE / 'train_0000.parquet'
TEST_PATH = CACHE / 'test_0000.parquet'

N_TRAIN = 2500
N_TEST = 500
N_TOPICS = 8
N_TOP_WORDS = 12
RANDOM_STATE = 42


def ensure_file(url: str, path: Path) -> None:
    if path.exists() and path.stat().st_size > 0:
        return
    print('Скачиваю', path.name, '...')
    urllib.request.urlretrieve(url, path)
    print('Готово:', path, 'байт:', path.stat().st_size)


ensure_file(TRAIN_URL, TRAIN_PATH)
ensure_file(TEST_URL, TEST_PATH)

train_full = pd.read_parquet(TRAIN_PATH)
test_full = pd.read_parquet(TEST_PATH)
print('Столбцы train:', train_full.columns.tolist())
print('Размеры полные:', len(train_full), len(test_full))

# текст новости: в mlsum/ru обычно text / summary / title
text_col = 'text' if 'text' in train_full.columns else train_full.columns[0]
train_df = train_full[[text_col]].dropna().head(N_TRAIN).copy()
test_df = test_full[[text_col]].dropna().head(N_TEST).copy()
train_df.columns = ['text']
test_df.columns = ['text']
print('Подвыборка train/test:', len(train_df), len(test_df))
print(train_df['text'].iloc[0][:240])


Столбцы train: ['text', 'summary', 'topic', 'url', 'title', 'date']
Размеры полные: 25556 757
Подвыборка train/test: 2500 500
Сладострастник в течение трех лет преследовал подростка в надежде совратить его. Как сообщили “МК” в следственном отделе по Хорошевскому району СУ СК при Прокуратуре РФ по Москве, 26 августа 2006 года 13-летний Павел вместе с другом отдыхал


## 2. Предобработка и векторизация

Очистка, токены; CountVectorizer / TfidfVectorizer для sklearn; словарь gensim — по тем же токенам.

In [2]:
# стоп-слова: общие знания + лекция preprocessing (без обязательного NLTK download в Colab)
RU_STOP = {
    'и', 'в', 'во', 'не', 'что', 'он', 'на', 'я', 'с', 'со', 'как', 'а', 'то', 'все', 'она',
    'так', 'его', 'но', 'да', 'ты', 'к', 'у', 'же', 'вы', 'за', 'бы', 'по', 'только', 'ее',
    'мне', 'было', 'вот', 'от', 'меня', 'еще', 'нет', 'о', 'из', 'ему', 'теперь', 'когда',
    'даже', 'ну', 'вдруг', 'ли', 'если', 'уже', 'или', 'ни', 'быть', 'был', 'него', 'до',
    'вас', 'нибудь', 'опять', 'уж', 'вам', 'ведь', 'там', 'потом', 'себя', 'ничего', 'ей',
    'может', 'они', 'тут', 'где', 'есть', 'надо', 'ней', 'для', 'мы', 'тебя', 'их', 'чем',
    'была', 'сам', 'чтоб', 'без', 'будто', 'человек', 'чего', 'раз', 'тоже', 'себе', 'под',
    'жизнь', 'будет', 'ж', 'тогда', 'кто', 'этот', 'того', 'потому', 'этого', 'какой',
    'совсем', 'ним', 'здесь', 'этом', 'один', 'почти', 'мой', 'тем', 'чтобы', 'нее',
    'сейчас', 'были', 'куда', 'зачем', 'сказать', 'всех', 'никогда', 'сегодня', 'можно',
    'при', 'наконец', 'два', 'об', 'другой', 'хоть', 'после', 'над', 'больше', 'тот',
    'через', 'эти', 'нас', 'про', 'всего', 'них', 'какая', 'много', 'разве', 'три', 'эту',
    'моя', 'впрочем', 'хорошо', 'свою', 'этой', 'перед', 'иногда', 'лучше', 'чуть', 'том',
    'нельзя', 'такой', 'им', 'более', 'всегда', 'конечно', 'всю', 'между', 'это', 'также',
    'изза', 'из-за', 'года', 'году', 'лет', 'также', 'которые', 'который', 'которая',
}


def preprocess(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^а-яёa-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in RU_STOP and len(t) > 2]
    return ' '.join(tokens)


train_df['clean'] = train_df['text'].map(preprocess)
test_df['clean'] = test_df['text'].map(preprocess)
train_tokens = [doc.split() for doc in train_df['clean']]
test_tokens = [doc.split() for doc in test_df['clean']]

count_vec = CountVectorizer(max_df=0.7, min_df=5, max_features=5000, stop_words=list(RU_STOP))
X_train_counts = count_vec.fit_transform(train_df['clean'])
X_test_counts = count_vec.transform(test_df['clean'])
feature_names = np.array(count_vec.get_feature_names_out())

tfidf_vec = TfidfVectorizer(max_df=0.7, min_df=5, max_features=5000)
X_train_tfidf = tfidf_vec.fit_transform(train_df['clean'])
X_test_tfidf = tfidf_vec.transform(test_df['clean'])
tfidf_names = np.array(tfidf_vec.get_feature_names_out())

dictionary = corpora.Dictionary(train_tokens)
dictionary.filter_extremes(no_below=5, no_above=0.7, keep_n=5000)
corpus_train = [dictionary.doc2bow(t) for t in train_tokens]
corpus_test = [dictionary.doc2bow(t) for t in test_tokens]
print('BoW shape:', X_train_counts.shape, 'словарь gensim:', len(dictionary))


BoW shape: (2500, 5000) словарь gensim: 5000


## 3. LDA, NMF, SVD/LSA (sklearn и gensim)

По ходу работы: LDA (sklearn + gensim), NMF (sklearn + gensim), SVD/LSA (sklearn).

In [3]:
def show_topics(name: str, components: np.ndarray, words: np.ndarray, n_words=N_TOP_WORDS):
    print(f'\n=== {name} ===')
    for i, row in enumerate(components):
        top = words[row.argsort()[::-1][:n_words]]
        print(f'Topic {i}:', ', '.join(top))


# --- LDA sklearn ---
lda_sk = LatentDirichletAllocation(
    n_components=N_TOPICS, random_state=RANDOM_STATE, learning_method='batch', max_iter=15
)
lda_sk.fit(X_train_counts)
show_topics('LDA sklearn', lda_sk.components_, feature_names)

# --- NMF sklearn ---
nmf_sk = NMF(n_components=N_TOPICS, random_state=RANDOM_STATE, init='nndsvda', max_iter=400)
nmf_sk.fit(X_train_tfidf)
show_topics('NMF sklearn', nmf_sk.components_, tfidf_names)

# --- SVD / LSA sklearn ---
svd = TruncatedSVD(n_components=N_TOPICS, random_state=RANDOM_STATE)
svd.fit(X_train_tfidf)
show_topics('SVD/LSA sklearn', np.abs(svd.components_), tfidf_names)

# --- LDA gensim ---
lda_gs = models.LdaModel(
    corpus=corpus_train,
    id2word=dictionary,
    num_topics=N_TOPICS,
    random_state=RANDOM_STATE,
    passes=8,
    alpha='auto',
)
print('\n=== LDA gensim ===')
for i in range(N_TOPICS):
    terms = lda_gs.show_topic(i, topn=N_TOP_WORDS)
    print(f'Topic {i}:', ', '.join(w for w, _ in terms))

# --- NMF gensim (через TF-IDF корпус) ---
tfidf_gs = models.TfidfModel(corpus_train)
corpus_tfidf = tfidf_gs[corpus_train]
nmf_gs = models.Nmf(
    corpus=corpus_tfidf,
    num_topics=N_TOPICS,
    id2word=dictionary,
    random_state=RANDOM_STATE,
    passes=8,
)
print('\n=== NMF gensim ===')
for i in range(N_TOPICS):
    terms = nmf_gs.show_topic(i, topn=N_TOP_WORDS)
    print(f'Topic {i}:', ', '.join(w for w, _ in terms))

print('\nМодели обучены: LDA/NMF/SVD sklearn + LDA/NMF gensim.')



=== LDA sklearn ===
Topic 0: россии, президент, президента, страны, россия, сша, очень, власти, время, вопрос, стране, государства
Topic 1: детей, время, очень, ребенка, нам, день, говорит, родители, отец, дети, жизни, дома
Topic 2: очень, просто, мира, время, россии, вообще, сборной, хотя, жизни, нам, день, спорта
Topic 3: летний, мужчина, стало, несколько, удалось, известно, время, однако, порядка, около, милиции, улице
Topic 4: войны, время, рублей, квартиры, победы, район, армии, млн, однако, тысяч, квартиру, дом
Topic 5: рублей, человека, метро, животных, тысяч, собак, несколько, могут, ученые, специалисты, людей, время
Topic 6: будут, рублей, области, руб, москвы, должны, москве, тысяч, год, время, тыс, кроме
Topic 7: суд, дело, мвд, суда, дела, адвокат, милиции, гибдд, решение, машины, суде, дтп



=== NMF sklearn ===
Topic 0: очень, просто, жизни, вообще, люди, кино, время, нам, хотя, почему, фильм, говорит
Topic 1: мужчина, летний, удалось, несколько, улице, известно, стало, милиции, около, порядка, милиционеры, овд
Topic 2: будут, области, сообщили, станут, должны, москвы, кроме, московской, территории, города, смогут, известно
Topic 3: сборной, мира, россии, спорта, команды, команда, тренер, чемпионата, европы, игры, матч, чемпионат
Topic 4: руб, тыс, млн, млрд, стоимость, рублей, цены, стоимости, бюджета, цен, тарифы, услуг
Topic 5: детей, ребенка, родители, школы, родителей, ребенок, дети, школе, мать, мальчика, мама, образования
Topic 6: рублей, суд, тыс, тысяч, дело, уголовное, деньги, сумму, отношении, свободы, прошлого, задержали
Topic 7: россии, президент, россия, президента, страны, сша, государства, власти, стране, премьер, медведев, вопрос

=== SVD/LSA sklearn ===
Topic 0: время, очень, россии, рублей, несколько, однако, будут, день, просто, дело, стало, области
To


=== LDA gensim ===
Topic 0: детей, будут, школы, образования, суд, рублей, москвы, области, закон, егэ, москве, должны
Topic 1: рублей, руб, млн, тыс, млрд, деньги, москвы, тысяч, человека, однако, будут, около
Topic 2: очень, время, просто, жизни, люди, вообще, хотя, нам, почему, день, могу, людей
Topic 3: время, несколько, дело, дома, однако, летний, удалось, милиции, мужчина, день, стало, вместе
Topic 4: время, метро, день, нам, очень, станции, несколько, именно, работы, внимание, стоит, площади
Topic 5: россии, мира, президент, россия, спорта, страны, время, будут, президента, москвы, федерации, сборной
Topic 6: россии, очень, могут, время, хотя, просто, деньги, будут, власти, именно, нужно, люди
Topic 7: сборной, мира, просто, команда, время, матч, тренер, матча, игры, команды, матче, юар



=== NMF gensim ===
Topic 0: будут, станут, должны, тысяч, закон, обязаны, случае, смогут, кроме, специалисты, помощи, средств
Topic 1: ребенка, женщина, отец, мать, мама, дочь, вместе, домой, метро, дома, сын, молодой
Topic 2: рублей, мужчина, летний, авто, млн, водитель, дтп, деньги, машины, долларов, задержали, гибдд
Topic 3: войны, победы, великой, удалось, отечественной, дома, ссср, днем, войне, около, обнаружили, территории
Topic 4: мира, спорта, россии, сборной, европы, чемпионат, федерации, спортсменов, игр, чемпионате, ванкувере, чемпион
Topic 5: сборной, цска, матча, матч, матче, команда, игры, команды, динамо, игроков, мвд, чемпионов
Topic 6: очень, просто, люди, нам, вообще, почему, жизни, хотя, знаю, сказал, нужно, кино
Topic 7: тыс, суд, руб, области, детей, рублей, московской, москвы, ребенка, сообщили, школы, уголовное

Модели обучены: LDA/NMF/SVD sklearn + LDA/NMF gensim.


## 4. Интерпретация тем

Кратко сопоставляем темы с сюжетами новостей; на тесте смотрим распределение LDA sklearn.

In [4]:
topic_mass = lda_sk.transform(X_test_counts).mean(axis=0)
order = topic_mass.argsort()[::-1]
print('Средняя доля тем на тесте (LDA sklearn), по убыванию:')
for i in order:
    top_words = feature_names[lda_sk.components_[i].argsort()[::-1][:8]]
    print(f'  topic {i}: mass={topic_mass[i]:.3f}; words={", ".join(top_words)}')

print(
    '\nИнтерпретация (эвристика по топ-словам): темы обычно группируются вокруг '
    'политики/власти, экономики, происшествий, спорта, культуры — конкретные '
    'лейблы зависят от подвыборки mlsum/ru. Сравнивая sklearn и gensim LDA, '
    'ожидаем похожие кластеры лексики при разной нумерации тем.'
)

# sanity: предсказание доминирующей темы для пары тестовых документов
dom = lda_sk.transform(X_test_counts[:3]).argmax(axis=1)
for i, t in enumerate(dom):
    print(f'test doc {i}: dominant topic {t}; snippet: {test_df["clean"].iloc[i][:120]}...')


Средняя доля тем на тесте (LDA sklearn), по убыванию:
  topic 2: mass=0.293; words=очень, просто, мира, время, россии, вообще, сборной, хотя
  topic 0: mass=0.223; words=россии, президент, президента, страны, россия, сша, очень, власти
  topic 6: mass=0.202; words=будут, рублей, области, руб, москвы, должны, москве, тысяч
  topic 1: mass=0.142; words=детей, время, очень, ребенка, нам, день, говорит, родители
  topic 7: mass=0.060; words=суд, дело, мвд, суда, дела, адвокат, милиции, гибдд
  topic 3: mass=0.033; words=летний, мужчина, стало, несколько, удалось, известно, время, однако
  topic 4: mass=0.027; words=войны, время, рублей, квартиры, победы, район, армии, млн
  topic 5: mass=0.020; words=рублей, человека, метро, животных, тысяч, собак, несколько, могут

Интерпретация (эвристика по топ-словам): темы обычно группируются вокруг политики/власти, экономики, происшествий, спорта, культуры — конкретные лейблы зависят от подвыборки mlsum/ru. Сравнивая sklearn и gensim LDA, ожидаем пох